## Query Rewriting

Users in enterprise settings rarely write retrieval-optimal queries. They write queries the same way they would ask a knowledgeable colleague: casually, with abbreviations, implicit context, and compound intentions. This is entirely reasonable human behavior, but it creates serious problems for embedding-based retrieval.

Consider these real-world query patterns:

- **"Q3 rev breakdown"** -- abbreviations that the embedding model may not handle well. Does "rev" mean revenue? Review? Revision?
- **"Compare the two ride-share companies"** -- no explicit mention of Uber or Lyft, no time period specified, no indication of what metrics to compare.
- **"How's the Lambda scaling and what about the timeout stuff?"** -- two distinct questions concatenated into a single query.
- **"Same as last time but for Lyft"** -- requires conversation history to resolve what "same as last time" refers to.

The core issue is how embedding models work. They convert text to fixed-size vectors by aggregating token representations across the entire input. A vague, multi-part query produces a vague, averaged embedding that sits in a general region of vector space without being particularly close to any specific document chunk. The embedding for "Q3 rev breakdown" lands somewhere between revenue documents, review documents, and revision documents -- close enough to all of them to retrieve a noisy mix but not close enough to any single category to retrieve precisely.

Query rewriting uses an LLM to transform the user's raw input into one or more optimized queries before they reach the retrieval layer. This is a pure pre-retrieval transformation. It requires no changes to the document index, the embedding model, or the retrieval infrastructure. You are simply giving the existing retrieval pipeline a better query to work with.

In [ ]:
import json
import re

from enterprise_rag.llm_model import llm


def rewrite_query(
    user_query: str,
    conversation_history: list = None
) -> str:
    history_context = ""
    if conversation_history:
        history_context = "\n".join(
            f"Q: {q}\nA: {a[:200]}..."
            for q, a in conversation_history[-3:]
        )
    rewrite_prompt = f'''
    You are a search query optimizer. Rephrase the user's entire
    query into a single, precise, retrieval-friendly question --
    do not just expand abbreviations in place, rewrite the full
    question so it stands on its own.
 
    Rules:
    1. Expand abbreviations ("Q3" -> "third quarter",
       "rev" -> "revenue")
    2. Replace vague references with specific terms using
       conversation history
    3. Add relevant domain context (year, company name)
       when clearly implied
    4. Do NOT add constraints the user did not express
    5. Return ONLY the rewritten query, no explanation
 
    Conversation history:
    {history_context if history_context else "None"}
 
 
    Original query: {user_query}
    Rewritten query:
    '''
    response = llm.invoke(rewrite_prompt)
    return response.content.strip()


def decompose_query(user_query: str) -> list:
    decompose_prompt = f'''
    Analyze the following query and determine if it contains
    multiple distinct information needs. If it does, break it
    into 2-4 focused atomic sub-queries. If it is already a
    single focused question, return it unchanged.
 
    Rules:
    - Each sub-query must be independently answerable
    - Sub-queries should not overlap or repeat each other
    - Preserve specific entities (company names, time periods)
    - Return ONLY a JSON array of strings
 
    Query: {user_query}
    '''
    response = llm.invoke(decompose_prompt)
    try:
        json_match = re.search(
            r"\[.*\]", response.content, re.DOTALL
        )
        return json.loads(json_match.group())
    except (json.JSONDecodeError, AttributeError):
        return [user_query]

In [3]:
rewrite_query("Hows the Lambda scaling and what about the timeout stuff?")

'How does AWS Lambda scaling work and what are the timeout limitations?'